In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from ira.ingest.ingest_scorecard import save
from ira.clean.clean_scorecard import clean
from ira.config import SCORECARD_KEY, BASE_URL

cwd = Path.cwd()
print(cwd)
root=cwd/'institutional-roi-analysis'
pd.set_option("display.max_columns",None)
display(root)

C:\Users\sebas\PycharmProjects\Git\Seb_branch


WindowsPath('C:/Users/sebas/PycharmProjects/Git/Seb_branch/institutional-roi-analysis')

In [2]:
tdf = pd.read_parquet(root/"data"/"raw"/"scorecard"/"national_scorecard.parquet")
display(tdf.head())
display(tdf.info())

,code,title,unit_id,distance,school.type,credential.level,earnings.1_yr.overall_median_earnings,earnings.1_yr.working_not_enrolled.overall_count,earnings.4_yr.overall_median_earnings,earnings.4_yr.working_not_enrolled.overall_count,earnings.5_yr.overall_median_earnings,earnings.5_yr.working_not_enrolled.overall_count,id,school.name,school.state,location.lat,location.lon,latest.school.locale,latest.school.carnegie_size_setting,latest.admissions.admission_rate.overall,latest.student.demographics.median_family_income,latest.student.students_with_pell_grant,latest.school.open_admissions_policy,latest.student.demographics.age_entry,latest.school.title_iv.eligibility_type,latest.admissions.sat_scores.average.overall,latest.admissions.act_scores.midpoint.cumulative,latest.student.grad_students
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,100654,Alabama A & M University,AL,34.783368,-86.568502,12,14,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                                            Non-Null Count  Dtype  
---  ------                                            --------------  -----  
 0   code                                              55930 non-null  object 
 1   title                                             55930 non-null  object 
 2   unit_id                                           55930 non-null  int64  
 3   distance                                          55930 non-null  int64  
 4   school.type                                       55930 non-null  object 
 5   credential.level                                  55930 non-null  int64  
 6   earnings.1_yr.overall_median_earnings             46006 non-null  float64
 7   earnings.1_yr.working_not_enrolled.overall_count  46006 non-null  float64
 8   earnings.4_yr.overall_median_earnings             55930 non-null  int64  
 9   earnings.4_yr.wor

None

# Why so many missing Admission Rates?

In [3]:
df = tdf.copy()
round(df.isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.3301)

In [4]:
round(df[df["latest.school.open_admissions_policy"]==2].isnull()["latest.admissions.admission_rate.overall"].mean(),4)

np.float64(0.005)

Approximately 40% of institutions have missing values for admission_rate.overall.
According to IPEDS reporting rules, institutions with an open admissions policy do not report traditional selectivity metrics such as admission rate or standardized test scores.

A check confirms that nearly all non-open-admission institutions report admission rates, indicating that the missingness is structural rather than random.

Therefore, missing admission rates are interpreted as corresponding primarily to open-admission institutions, and the open_admissions_policy variable is retained to preserve this structural distinction.

In [5]:
df = df.drop(columns="id")
df = clean(df)

In [6]:
df["selectivity_bucket"] = pd.cut(
    df["admission_rate_overall"],
    bins=[0, 0.3, 0.7, 1],
    labels=["elite", "mid", "open"]
)

In [7]:
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55930 entries, 0 to 55929
Data columns (total 28 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   code                            55930 non-null  string  
 1   title                           55930 non-null  string  
 2   unit_id                         55930 non-null  string  
 3   distance                        55930 non-null  int64   
 4   school_type                     55930 non-null  string  
 5   credential_level                55930 non-null  int64   
 6   1_yr_median_earnings            46006 non-null  float64 
 7   1_yr_working_count              46006 non-null  float64 
 8   4_yr_median_earnings            55930 non-null  int64   
 9   4_yr_working_count              55930 non-null  int64   
 10  5_yr_median_earnings            41343 non-null  float64 
 11  5_yr_working_count              41343 non-null  float64 
 12  school_name       

None

In [8]:
df.head()

,code,title,unit_id,distance,school_type,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,school_name,school_state,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students,selectivity_bucket
0,0305,Forestry.,100654,1,Public,3,NaN,NaN,64749,16,NaN,NaN,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
1,1002,Audiovisual Communications Technologies/Techni...,100654,1,Public,3,28938.0,31.0,42272,37,NaN,NaN,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
2,1101,"Computer and Information Sciences, General.",100654,1,Public,3,63900.0,29.0,88490,39,85218.0,27.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
3,1312,Teacher Education and Professional Development...,100654,2,Public,5,56295.0,21.0,60412,24,69062.0,18.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid
4,1410,"Electrical, Electronics, and Communications En...",100654,1,Public,3,72241.0,40.0,98045,43,90409.0,29.0,Alabama A & M University,AL,34.783368,-86.568502,12,14.0,0.5795,23553.0,0.852793,2.0,20.0,1,938.0,18.0,925.0,mid


In [9]:
save(df,file_type="scorecard",clean=0,file_name="national_preprocessed_scorecard_programs")

In [10]:
display("Relevant numeric variables statistics:", df.select_dtypes(exclude=object).describe())
display("Missing values per column:", df.isna().sum())
display("Correlation matrix:", df.corr(numeric_only=True))

'Relevant numeric variables statistics:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
count,55930.000000,55930.000000,46006.000000,46006.000000,55930.000000,55930.000000,41343.000000,41343.000000,55930.000000,55930.000000,55930.000000,51620.000000,37469.0,55566.000000,53574.0,54911.000000,55566.000000,55930.000000,28424.000000,25623.000000,39457.000000
mean,1.298301,3.423368,49347.405447,110.133591,65571.535759,101.731164,63604.452483,108.088358,37.955487,-89.541457,18.530753,11.347792,0.720972,42843.610319,0.633126,1.685819,23.105226,1.018309,1209.583697,25.467275,4507.224345
std,0.739933,5.726003,24051.517980,300.209665,27720.152721,280.310977,28396.672776,278.890877,5.436187,15.649505,9.133195,4.841543,0.2318,22801.900981,0.176964,0.464193,3.379662,0.228855,143.918529,4.015281,6172.442970
min,0.000000,1.000000,4506.000000,16.000000,8305.000000,16.000000,7826.000000,16.000000,-14.322636,-170.742774,11.000000,1.000000,0.0,0.000000,0.114961,1.000000,17.000000,1.000000,720.000000,14.000000,1.000000
25%,1.000000,2.000000,32459.250000,28.000000,47566.000000,25.000000,45549.000000,28.000000,34.152076,-96.581077,11.000000,9.000000,0.6154,24244.000000,0.486853,1.000000,21.000000,1.000000,1099.000000,23.000000,725.000000
50%,1.000000,3.000000,44141.500000,47.000000,59562.500000,43.000000,57882.000000,47.000000,39.328977,-85.533519,13.000000,13.000000,0.7841,37667.500000,0.62919,2.000000,22.000000,1.000000,1190.000000,25.000000,2264.000000
75%,1.000000,3.000000,61827.750000,98.000000,77911.750000,91.000000,75760.500000,97.000000,41.703058,-78.157433,21.000000,15.000000,0.8901,58609.000000,0.777258,2.000000,25.000000,1.000000,1297.000000,28.000000,5711.000000
max,3.000000,99.000000,272682.000000,11263.000000,336392.000000,9665.000000,384547.000000,10468.000000,64.857560,145.721733,43.000000,18.000000,1.0,179864.000000,0.99674,2.000000,48.000000,5.000000,1560.000000,35.000000,55120.000000


'Missing values per column:'

code                                  0
title                                 0
unit_id                               0
distance                              0
school_type                           0
credential_level                      0
1_yr_median_earnings               9924
1_yr_working_count                 9924
4_yr_median_earnings                  0
4_yr_working_count                    0
5_yr_median_earnings              14587
5_yr_working_count                14587
school_name                           0
school_state                          0
location_lat                          0
location_lon                          0
locale                                0
carnegie_size_setting              4310
admission_rate_overall            18461
median_family_income                364
students_with_pell_grant           2356
open_admissions_policy             1019
age_entry                           364
title_iv_eligibility_type             0
sat_scores_average_overall        27506


'Correlation matrix:'

,distance,credential_level,1_yr_median_earnings,1_yr_working_count,4_yr_median_earnings,4_yr_working_count,5_yr_median_earnings,5_yr_working_count,location_lat,location_lon,locale,carnegie_size_setting,admission_rate_overall,median_family_income,students_with_pell_grant,open_admissions_policy,age_entry,title_iv_eligibility_type,sat_scores_average_overall,act_scores_midpoint_cumulative,grad_students
distance,1.000000,-0.064231,0.087875,0.045212,0.026595,0.060110,0.011334,0.049668,-0.023778,0.009573,0.027418,-0.029884,0.090494,-0.100275,0.084193,-0.063824,0.246856,0.149040,-0.125451,-0.139590,0.066216
credential_level,-0.064231,1.000000,0.528689,-0.049601,0.127210,-0.022065,0.542649,-0.059269,-0.005253,-0.013301,-0.048364,0.127033,-0.021647,0.075802,-0.081029,0.117613,-0.045270,0.001576,0.025357,0.024428,0.092013
1_yr_median_earnings,0.087875,0.528689,1.000000,0.032196,0.912971,0.012966,0.871953,-0.004042,0.108403,-0.003562,-0.089905,0.216672,-0.185652,0.226013,-0.261235,0.237967,-0.088883,0.020126,0.243557,0.240116,0.163279
1_yr_working_count,0.045212,-0.049601,0.032196,1.000000,0.015639,0.972806,0.020124,0.826064,-0.054290,-0.041894,-0.056637,-0.041627,0.027674,-0.089292,0.078346,-0.092510,0.178741,0.072431,0.023925,0.051733,0.143269
4_yr_median_earnings,0.026595,0.127210,0.912971,0.015639,1.000000,0.013077,0.943206,-0.011333,0.135369,-0.004781,-0.134777,0.336668,-0.277068,0.330410,-0.349991,0.335767,-0.202435,0.008376,0.365682,0.364367,0.217848
4_yr_working_count,0.060110,-0.022065,0.012966,0.972806,0.013077,1.000000,0.013833,0.884454,-0.051139,-0.033560,-0.057614,-0.038005,0.030593,-0.087938,0.079297,-0.089302,0.177095,0.086418,0.016139,0.045802,0.117354
5_yr_median_earnings,0.011334,0.542649,0.871953,0.020124,0.943206,0.013833,1.000000,-0.014781,0.132200,-0.006525,-0.124917,0.342259,-0.284752,0.339342,-0.364824,0.354463,-0.223239,0.002439,0.374018,0.374147,0.209618
5_yr_working_count,0.049668,-0.059269,-0.004042,0.826064,-0.011333,0.884454,-0.014781,1.000000,-0.061587,-0.050599,-0.061436,-0.071796,0.042269,-0.116831,0.112136,-0.122692,0.206762,0.059092,0.008662,0.059848,0.083087
location_lat,-0.023778,-0.005253,0.108403,-0.054290,0.135369,-0.051139,0.132200,-0.061587,1.000000,0.021273,0.108338,0.010766,0.108251,0.359370,-0.365666,0.095307,-0.113198,-0.023495,0.145722,0.151120,-0.041482
location_lon,0.009573,-0.013301,-0.003562,-0.041894,-0.004781,-0.033560,-0.006525,-0.050599,0.021273,1.000000,0.100717,0.006911,-0.107101,0.189199,-0.211509,0.082897,-0.140627,0.032241,0.138953,0.196015,-0.038194


In [11]:
from sklearn.model_selection import GridSearchCV, KFold, train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score

# Reproducibility
RANDOM_STATE = 42

In [12]:
model_df = df.copy()
target = '4_yr_median_earnings'

drop_columns=["title","4_yr_working_count","school_name",'1_yr_median_earnings',
       '1_yr_working_count', '4_yr_median_earnings', '4_yr_working_count',
       '5_yr_median_earnings', '5_yr_working_count']
X = model_df.drop(columns=[target]).copy()
X = X.drop(columns=[],errors="ignore")
y = pd.to_numeric(model_df[target], errors='coerce').copy()
y_log = np.log10(y)

cat_cols = [
    'code',
    'school_type',
    'locale',
    'carnegie_size_setting',
    'open_admissions_policy',
    'title_iv_eligibility_type',
    'credential_level',
    'distance',
    "selectivity_bucket"
]
num_cols = [
    'admission_rate_overall',
    "location_lat",
    "location_lon", 
    'median_family_income',
    'students_with_pell_grant',
    'age_entry'
    
]

# categorical: force plain object and replace missing with np.nan
for c in cat_cols:
    X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})
    X[c] = X[c].astype(str).replace({"nan": "Missing", "<NA>": "Missing"})

for c in cat_cols:
    X[c] = X[c].astype(object)
    X[c] = X[c].replace({pd.NA: np.nan})

# numeric: force numeric with np.nan
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

# nuclear option: remove any lingering pd.NA anywhere in X
X = X.astype(object).replace({pd.NA: np.nan})

# now restore numeric cols back to numeric dtype
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors='coerce')

C:\Users\sebas\AppData\Local\Temp\ipykernel_20984\2737674556.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = X.astype(object).replace({pd.NA: np.nan})


In [13]:
X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log,
    test_size=0.2,
    random_state=RANDOM_STATE
)
# fix: coerce cat cols to string AFTER imputation
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        missing_values=np.nan, 
        strategy="constant", 
        fill_value="Missing"
    )),
    ("to_str", FunctionTransformer(
        lambda X: X.astype(str),  # ← force everything to string after imputing
        feature_names_out="one-to-one"  # ← add this
    )),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols),
])

In [14]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', Ridge())
])

param_grid = {
    'reg__alpha': [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid.fit(X_train, y_train_log)

print("Best params:", grid.best_params_)
print("Best CV MAE:", round(-grid.best_score_, 2))
best_model = grid.best_estimator_
preds = best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, preds), 2))
print("Test R2:", round(r2_score(y_test_log, preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.1}
Best CV MAE: 0.06
Test MAE: 0.06
Test R2: 0.766


In [16]:
feature_names = best_model.named_steps["preprocessor"].get_feature_names_out()

# Ridge uses coef_ not feature_importances_
coefficients = best_model.named_steps["reg"].coef_

importance = pd.Series(coefficients, index=feature_names)

# sort by absolute value to see most influential features
importance = importance.reindex(importance.abs().sort_values(ascending=False).index)

print("Most important features:")
print(importance.head(20).round(4).to_string())

Most important features:
cat__code_5105    0.3761
cat__code_4102    0.3249
cat__code_5133   -0.3147
cat__code_6001    0.3137
cat__code_5114    0.2998
cat__code_5134   -0.2691
cat__code_4103    0.2647
cat__code_5104    0.2551
cat__code_4503   -0.2511
cat__code_3014   -0.2482
cat__code_1509    0.2444
cat__code_2805    0.2343
cat__code_1437    0.2302
cat__code_4903    0.2262
cat__code_1425    0.2258
cat__code_1099   -0.2171
cat__code_5005   -0.2157
cat__code_1514    0.2122
cat__code_1409    0.2101
cat__code_3022   -0.2098


In [17]:
ls_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", Lasso(max_iter=100000, random_state=RANDOM_STATE))
])

ls_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0]
}

ls_grid = GridSearchCV(
    ls_pipe,
    param_grid=ls_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

ls_grid.fit(X_train, y_train_log)

print("Best params:", ls_grid.best_params_)
print("Best CV MAE:", round(-ls_grid.best_score_, 2))

ls_best_model = ls_grid.best_estimator_
ls_preds = ls_best_model.predict(X_test)

print("Test MAE:", round(mean_absolute_error(y_test_log, ls_preds), 2))
print("Test R2:", round(r2_score(y_test_log, ls_preds), 4))

Fitting 5 folds for each of 6 candidates, totalling 30 fits
Best params: {'reg__alpha': 0.01}
Best CV MAE: 0.12
Test MAE: 0.11
Test R2: 0.2705


In [18]:
enet_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", ElasticNet(max_iter=100000))
])


enet_param_grid = {
    "reg__alpha": [0.01, 0.1, 1.0, 3.0, 10.0, 50.0],
    "reg__l1_ratio": [0.005, 0.1, 0.3, 0.5, 0.7, 0.9]
}

enet_grid = GridSearchCV(
    enet_pipe,
    param_grid=enet_param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score="raise"
)

enet_grid.fit(X_train, y_train_log)

print("ElasticNet Best params:", enet_grid.best_params_)
print("ElasticNet Best CV MAE:", round(-enet_grid.best_score_, 2))
enet_best_model = enet_grid.best_estimator_
enet_preds = enet_best_model.predict(X_test)

print("ElasticNet Test MAE:", round(mean_absolute_error(y_test_log, enet_preds), 2))
print("ElasticNet Test R2:", round(r2_score(y_test_log, enet_preds), 4))

Fitting 5 folds for each of 36 candidates, totalling 180 fits
ElasticNet Best params: {'reg__alpha': 0.01, 'reg__l1_ratio': 0.005}
ElasticNet Best CV MAE: 0.08
ElasticNet Test MAE: 0.08
ElasticNet Test R2: 0.6305


In [19]:
baseline_pred = [y_train_log.mean()] * len(y_test_log)

print("Baseline MAE:", round(mean_absolute_error(y_test_log, baseline_pred), 2))

Baseline MAE: 0.13


In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict


# pipe_rfr = Pipeline([
#     ('preprocessor', preprocessor),
#     ('clf', RandomForestRegressor(
#         random_state=RANDOM_STATE,
#         n_jobs=-1
#     ))
# ])

# param_grid_rfr = {
#     'clf__n_estimators': [100, 200, 300],
#     'clf__max_depth': [None, 5, 10, 20],
#     'clf__min_samples_split': [2, 5, 10],
#     'clf__min_samples_leaf': [1, 2, 4]
# }

# grid_rfr = GridSearchCV(
#     estimator=pipe_rfr,
#     param_grid=param_grid_rfr,
#     scoring='neg_mean_absolute_error',
#     cv=cv,
#     n_jobs=1,
#     refit=True,
#     verbose=1,
#     error_score='raise'
# )

# grid_rfr.fit(X_train, y_train_log)

# print("RF Best params:", grid_rfr.best_params_)
# print("RF Best CV MAE:", round(-grid_rfr.best_score_, 2))

# best_rfr = grid_rfr.best_estimator_
# rfr_preds = best_rfr.predict(X_test)

# print("RF Test MAE:", round(mean_absolute_error(y_test_log, rfr_preds), 2))
# print("RF Test R2:", round(r2_score(y_test_log, rfr_preds), 4))

In [20]:
from xgboost import XGBRegressor

pipe_xgb = Pipeline([
    ('preprocessor', preprocessor),
    ('reg', XGBRegressor(
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0
    ))
])

param_grid_xgb = {
    'reg__n_estimators': [200, 300],
    'reg__max_depth': [5, 10], 
    'reg__learning_rate': [0.05, 0.1],
    'reg__subsample': [0.8, 1.0],
}

grid_xgb_log = GridSearchCV(
    estimator=pipe_xgb,
    param_grid=param_grid_xgb,
    scoring='neg_mean_absolute_error',
    cv=cv,
    n_jobs=1,
    refit=True,
    verbose=1,
    error_score='raise'
)

grid_xgb_log.fit(X_train, y_train_log)

print("XGB Best params:", grid_xgb_log.best_params_)
print("XGB Best CV MAE:", round(-grid_xgb_log.best_score_, 2))

best_xgb_log = grid_xgb_log.best_estimator_
xgb_preds_log = best_xgb_log.predict(X_test)

print("XGB Test MAE:", round(mean_absolute_error(y_test_log, xgb_preds_log), 2))
print("XGB Test R2:", round(r2_score(y_test_log, xgb_preds_log), 4))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE: 0.05
XGB Test MAE: 0.05
XGB Test R2: 0.8401


Fitting 5 folds for each of 16 candidates, totalling 80 fits
XGB Best params: {'reg__learning_rate': 0.1, 'reg__max_depth': 10, 'reg__n_estimators': 300, 'reg__subsample': 0.8}
XGB Best CV MAE: 0.05
XGB Test MAE: 0.05
XGB Test R2: 0.8325

All models significantly outperformed the baseline. The linear model and Random Forest performed similarly, suggesting that linear relationships explain a large portion of the variance. However, XGBoost achieved the best performance, reducing MAE from 0.14 to 0.13 and increasing R2 to ~0.83. This indicates that nonlinear interactions exist in the data and are effectively captured by gradient boosting methods.

In [21]:
feature_names = best_xgb_log.named_steps["preprocessor"].get_feature_names_out()

importances = best_xgb_log.named_steps["reg"].feature_importances_

importance = pd.Series(importances, index=feature_names)

print("most important features")
display(importance.sort_values(ascending=False).head(15))
print("least important features")
display(importance.sort_values(ascending=True).head(15))

most important features


cat__code_1204                     0.072218
cat__open_admissions_policy_1.0    0.050644
cat__code_5138                     0.036866
cat__selectivity_bucket_elite      0.020410
cat__code_5007                     0.015196
cat__code_1107                     0.014698
cat__code_1410                     0.013977
cat__credential_level_7            0.013495
cat__code_1409                     0.011405
cat__code_1419                     0.011257
cat__code_1407                     0.011232
cat__code_1312                     0.009810
cat__code_1101                     0.009771
cat__code_5213                     0.009697
cat__code_5208                     0.009350
dtype: float32

least important features


cat__code_0199                      0.0
cat__title_iv_eligibility_type_3    0.0
cat__code_5116                      0.0
cat__code_0499                      0.0
cat__code_1099                      0.0
cat__code_0999                      0.0
cat__code_1302                      0.0
cat__code_1299                      0.0
cat__code_5137                      0.0
cat__code_5136                      0.0
cat__code_1307                      0.0
cat__code_1306                      0.0
cat__code_4899                      0.0
cat__code_1604                      0.0
cat__code_1612                      0.0
dtype: float32

In [22]:
print("School level n:", X['unit_id'].nunique())
print('Program x Creds x Schools rows:', len(X))

School level n: 4967
Program x Creds x Schools rows: 55930


Feature importance analysis from the XGBoost model shows that categorical variables, particularly program codes (CIP), credential level, admission policy, and carnegie classification, were the most influential predictors. Additionally, several features had zero importance, indicating that certain categories did not contribute meaningfully to prediction, likely due to lack of signal.

The model was trained on log-transformed earnings to address skewness and improve predictive stability. Predictions were then exponentiated back to the original scale to evaluate performance and interpret errors in dollar terms.

Predictions are the typical (median-like) expected earnings rather than the average, which is appropriate given the skewed nature of income data.